In [1]:
from dataset_utils import *
from experiment_runner import TaskDefinition
from feature_pipeline import FeaturePipeline, ProcessedFeaturePipeline
from experiment_runner import run_continual_experiment
import torch
import numpy as np
import random
from classifier_strategies import *

In [2]:
# General Setup
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Defining Tasks
TASK_DEFINITIONS = [
    TaskDefinition(
        name="Cresci17 Dataset",
        train_factory=lambda: ProcessedDataset("train", "../datasets/ProcessedDatasets/", "Cresci17"),
        test_factory=lambda: ProcessedDataset("test", "../datasets/ProcessedDatasets/", "Cresci17"),
        label_transform=lambda label: label,
    ),
    TaskDefinition(
        name="Caverlee 11 Dataset",
        train_factory= lambda: ProcessedDataset("train", "../datasets/ProcessedDatasets/", "Caverlee11"),
        test_factory= lambda: ProcessedDataset("test", "../datasets/ProcessedDatasets/", "Caverlee11"),
        label_transform= lambda label: label,
    ),
    TaskDefinition(
        name="Cresci 18 Dataset",
        train_factory= lambda: ProcessedDataset("train", "../datasets/ProcessedDatasets/", "Cresci18"),
        test_factory= lambda: ProcessedDataset("test", "../datasets/ProcessedDatasets/", "Cresci18"),
        label_transform= lambda label: label,
    ),
    TaskDefinition(
        name="TwiBot 20 Dataset",
        train_factory= lambda: ProcessedDataset("train", "../datasets/ProcessedDatasets/", "Twibot20"),
        test_factory= lambda: ProcessedDataset("test", "../datasets/ProcessedDatasets/", "Twibot20"),
        label_transform= lambda label: label,
    ),
    TaskDefinition(
        name="TwiBot 22 Dataset",
        train_factory= lambda: ProcessedDataset("train", "../datasets/ProcessedDatasets/", "Twibot22"),
        test_factory= lambda: ProcessedDataset("test", "../datasets/ProcessedDatasets/", "Twibot22"),
        label_transform= lambda label: label,
    ),
]

FEATURE_PIPELINE_CONFIG = {
    "embedding_model": "distilbert-base-uncased",

    "max_tweets_per_user": 300,
    "tweet_batch_size": 300,
    "max_token_length": 512,

    "umap_components": 38,
    "umap_neighbors": 20,
    "umap_min_dist": 0.1,
    "umap_metric": "cosine",

    "random_seed": RANDOM_SEED,
}


## Baseline Classifier

In [3]:
STRATEGY_CONFIG = {
    "epochs": 15,
    "learning_rate": 1e-2,
    "dropout_p": 0.1,

    "replay_per_class": 1000,
    "balanced_samples_per_class": 1000,

    "hdbscan_min_cluster_size": 10,
    "hdbscan_current_fraction": 0.80,

    # True = expand when labels show that a new class arrived.
    # HDBSCAN remains logged as a diagnostic.
    "use_intervention_override": True,

    "eval_batch_size": 512,
    "random_seed": RANDOM_SEED,
}


feature_pipeline = ProcessedFeaturePipeline(
    config=FEATURE_PIPELINE_CONFIG,
    device=device,
)

strategy = BaselineClassifierStrategy()

results = run_continual_experiment(
    task_definitions=TASK_DEFINITIONS,
    feature_pipeline=feature_pipeline,
    strategy=strategy,
    strategy_config=STRATEGY_CONFIG,
    device=device,
)

strategy_state = results["strategy_state"]
experiment_manager = results["experiment_manager"]
all_step_metrics = results["all_step_metrics"]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



STEP 0: Cresci17 Dataset
Embedding 'Cresci17 Dataset/train'...
label_mapping {0: 0, 1: 1, 2: 2, 3: 3}
Finished 'Cresci17 Dataset/train': (10145, 45), labels=[0, 1, 2, 3]
Current labels: ['0', '1', '2', '3']
Unseen labels: ['0', '1', '2', '3']
HDBSCAN novelty signal: False
Created classifier with 4 outputs.
Actual classifier outputs: 8
Balanced classifier-training counts:
  0:  1000
  1:  1000
  2:  1000
  3:  1000
Epoch [1/15] - Loss: 0.6667
Epoch [2/15] - Loss: 0.3542
Epoch [3/15] - Loss: 0.3202
Epoch [4/15] - Loss: 0.3173
Epoch [5/15] - Loss: 0.2982
Epoch [6/15] - Loss: 0.2769
Epoch [7/15] - Loss: 0.2815
Epoch [8/15] - Loss: 0.2702
Epoch [9/15] - Loss: 0.2654
Epoch [10/15] - Loss: 0.2565
Epoch [11/15] - Loss: 0.2629
Epoch [12/15] - Loss: 0.2462
Epoch [13/15] - Loss: 0.2622
Epoch [14/15] - Loss: 0.2539
Epoch [15/15] - Loss: 0.2385
Loaded best model weights (Best Loss: 0.2385)
Embedding 'Cresci17 Dataset/test'...
label_mapping {0: 0, 1: 1, 2: 2, 3: 3}
Finished 'Cresci17 Dataset/test':

## SVM Classifier

In [4]:
STRATEGY_CONFIG = {
    "epochs": 15,
    "learning_rate": 1e-2,
    "dropout_p": 0.1,

    "replay_per_class": 1000,
    "balanced_samples_per_class": 1000,

    "hdbscan_min_cluster_size": 10,
    "hdbscan_current_fraction": 0.80,

    # True = expand when labels show that a new class arrived.
    # HDBSCAN remains logged as a diagnostic.
    "use_intervention_override": True,

    "eval_batch_size": 512,
    "random_seed": RANDOM_SEED,
}


feature_pipeline = ProcessedFeaturePipeline(
    config=FEATURE_PIPELINE_CONFIG,
    device=device,
)

strategy = MultiSVMClassifierStrategy()

results = run_continual_experiment(
    task_definitions=TASK_DEFINITIONS,
    feature_pipeline=feature_pipeline,
    strategy=strategy,
    strategy_config=STRATEGY_CONFIG,
    device=device,
)

strategy_state = results["strategy_state"]
experiment_manager = results["experiment_manager"]
all_step_metrics = results["all_step_metrics"]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



STEP 0: Cresci17 Dataset
Embedding 'Cresci17 Dataset/train'...
label_mapping {0: 0, 1: 1, 2: 2, 3: 3}
Finished 'Cresci17 Dataset/train': (10145, 45), labels=[0, 1, 2, 3]
Current labels: ['0', '1', '2', '3']
Unseen labels: ['0', '1', '2', '3']
HDBSCAN novelty signal: False
Created classifier with 4 outputs.
Actual classifier outputs: 4
Balanced classifier-training counts:
  0:  1000
  1:  1000
  2:  1000
  3:  1000
Epoch [1/15] - Loss: 3.2949
Epoch [2/15] - Loss: 2.9857
Epoch [3/15] - Loss: 3.1422
Epoch [4/15] - Loss: 3.0145
Epoch [5/15] - Loss: 3.1095
-- stopping early --
Loaded best model weights (Best Loss: 2.9857)
Embedding 'Cresci17 Dataset/test'...
label_mapping {0: 0, 1: 1, 2: 2, 3: 3}
Finished 'Cresci17 Dataset/test': (1311, 45), labels=[0, 1, 2, 3]
Evaluating 1 test task(s)...
  Task 0 — Cresci17 Dataset: accuracy=0.3303, macro-F1=0.1241

--- Step 0 complete ---
    Avg Accuracy (this step): 0.3303
    Forgetting Measure:       0.0000
    Row scores: ['0.330']

=== Now startin

## Confidence Classifier

In [5]:
STRATEGY_CONFIG = {
    "epochs": 15,
    "learning_rate": 1e-2,
    "dropout_p": 0.1,

    "replay_per_class": 1000,
    "balanced_samples_per_class": 1000,

    "hdbscan_min_cluster_size": 10,
    "hdbscan_current_fraction": 0.80,

    # True = expand when labels show that a new class arrived.
    # HDBSCAN remains logged as a diagnostic.
    "use_intervention_override": True,

    "eval_batch_size": 512,
    "random_seed": RANDOM_SEED,
}


feature_pipeline = ProcessedFeaturePipeline(
    config=FEATURE_PIPELINE_CONFIG,
    device=device,
)

strategy = MultiClassConfidenceClassifierStrategy()

results = run_continual_experiment(
    task_definitions=TASK_DEFINITIONS,
    feature_pipeline=feature_pipeline,
    strategy=strategy,
    strategy_config=STRATEGY_CONFIG,
    device=device,
)

strategy_state = results["strategy_state"]
experiment_manager = results["experiment_manager"]
all_step_metrics = results["all_step_metrics"]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



STEP 0: Cresci17 Dataset
Embedding 'Cresci17 Dataset/train'...
label_mapping {0: 0, 1: 1, 2: 2, 3: 3}
Finished 'Cresci17 Dataset/train': (10145, 45), labels=[0, 1, 2, 3]
Current labels: ['0', '1', '2', '3']
Unseen labels: ['0', '1', '2', '3']
HDBSCAN novelty signal: False
Created classifier with 4 outputs.
Actual classifier outputs: 4
Balanced classifier-training counts:
  0:  1000
  1:  1000
  2:  1000
  3:  1000
Epoch [1/15] - Loss: 0.6308
Epoch [2/15] - Loss: 0.3487
Epoch [3/15] - Loss: 0.3066
Epoch [4/15] - Loss: 0.3113
Epoch [5/15] - Loss: 0.3118
Epoch [6/15] - Loss: 0.3177
-- stopping early --
Loaded best model weights (Best Loss: 0.3066)
Embedding 'Cresci17 Dataset/test'...
label_mapping {0: 0, 1: 1, 2: 2, 3: 3}
Finished 'Cresci17 Dataset/test': (1311, 45), labels=[0, 1, 2, 3]
Evaluating 1 test task(s)...
  Task 0 — Cresci17 Dataset: accuracy=0.2799, macro-F1=0.1983

--- Step 0 complete ---
    Avg Accuracy (this step): 0.2799
    Forgetting Measure:       0.0000
    Row scores

## PNN Classifier

In [6]:
STRATEGY_CONFIG = {
    "epochs": 15,
    "learning_rate": 1e-2,
    "dropout_p": 0.1,

    "replay_per_class": 1000,
    "balanced_samples_per_class": 1000,

    "hdbscan_min_cluster_size": 10,
    "hdbscan_current_fraction": 0.80,

    # True = expand when labels show that a new class arrived.
    # HDBSCAN remains logged as a diagnostic.
    "use_intervention_override": True,

    "eval_batch_size": 512,
    "random_seed": RANDOM_SEED,
}


feature_pipeline = ProcessedFeaturePipeline(config=FEATURE_PIPELINE_CONFIG,device=device)

strategy = MulticlassPNNStrategy()

results = run_continual_experiment(
    task_definitions=TASK_DEFINITIONS,
    feature_pipeline=feature_pipeline,
    strategy=strategy,
    strategy_config=STRATEGY_CONFIG,
    device=device,
)

strategy_state = results["strategy_state"]
experiment_manager = results["experiment_manager"]
all_step_metrics = results["all_step_metrics"]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



STEP 0: Cresci17 Dataset
Embedding 'Cresci17 Dataset/train'...
label_mapping {0: 0, 1: 1, 2: 2, 3: 3}
Finished 'Cresci17 Dataset/train': (10145, 45), labels=[0, 1, 2, 3]
Current labels: ['0', '1', '2', '3']
Unseen labels: ['0', '1', '2', '3']
HDBSCAN novelty signal: False
Created classifier with 4 outputs.
Actual classifier outputs: 4
Balanced classifier-training counts:
  0: 1000
  1: 1000
  2: 1000
  3: 1000
Epoch [1/15] - Loss: 0.6211
Epoch [2/15] - Loss: 0.3344
Epoch [3/15] - Loss: 0.3250
Epoch [4/15] - Loss: 0.3101
Epoch [5/15] - Loss: 0.2623
Epoch [6/15] - Loss: 0.2583
Epoch [7/15] - Loss: 0.2555
Epoch [8/15] - Loss: 0.2548
Epoch [9/15] - Loss: 0.2718
Epoch [10/15] - Loss: 0.2340
Epoch [11/15] - Loss: 0.2398
Epoch [12/15] - Loss: 0.2280
Epoch [13/15] - Loss: 0.2347
Epoch [14/15] - Loss: 0.2373
Epoch [15/15] - Loss: 0.2129
Loaded best model weights (Best Loss: 0.2129)
Embedding 'Cresci17 Dataset/test'...
label_mapping {0: 0, 1: 1, 2: 2, 3: 3}
Finished 'Cresci17 Dataset/test': (13